# ⚡ WavLM + Prosody Extraction — Kaggle P100 Fix
**Problem:** PyTorch 2.10+cu128 needs Hopper (sm_90), but Kaggle has P100 (sm_60)
**Solution:** Install PyTorch 1.13.1 with CUDA 11.7 (supports sm_60)

**This notebook:**
1. Install PyTorch 1.13.1 + CUDA 11.7
2. Load WavLM
3. Download audio from YouTube
4. Extract features
5. Save to output


In [ ]:
# Cell 1: Fix PyTorch for Kaggle P100
import subprocess
import os

print('=== Installing PyTorch 1.13.1 + CUDA 11.7 ===')

# Uninstall current PyTorch
subprocess.run(['pip', 'uninstall', 'torch', 'torchvision', '-y', '-q'], capture_output=True)

# Install PyTorch 1.13.1 with CUDA 11.7
result = subprocess.run([
    'pip', 'install',
    'torch==1.13.1',
    'torchvision==0.14.1',
    'torchaudio==0.13.1',
    '--index-url',
    'https://download.pytorch.org/whl/cu117'
], capture_output=True, text=True, timeout=300)

print('Return code:', result.returncode)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:] if result.stderr else 'none')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA version: {torch.version.cuda}')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    
    # Test GPU
    try:
        x = torch.randn(100, 100).cuda()
        y = x @ x
        print('GPU test: SUCCESS')
    except Exception as e:
        print(f'GPU test FAILED: {e}')

In [ ]:
# Cell 2: Load WavLM
from transformers import AutoModel
import torch

print('Loading WavLM...')
wavlm = AutoModel.from_pretrained('microsoft/wavlm-base')
wavlm.eval()
wavlm = wavlm.cuda()
print('WavLM loaded on GPU')

In [ ]:
# Cell 3: Setup
import subprocess
import os

os.makedirs('/kaggle/working/audio', exist_ok=True)
os.makedirs('/kaggle/working/features', exist_ok=True)

# Install yt-dlp
subprocess.run(['pip', 'install', 'yt-dlp', '-q'], capture_output=True)
print('Setup complete')

In [ ]:
# Cell 4: Download + Extract for first video
import numpy as np
import librosa
import torch

# Video list - first 5 for test
VIDEO_IDS = [
    'q112mLKiUCw', 'J9HLFJgUCW0', 'LWYfo_8t5WQ', 'oiRyNnyG698',
    'KVMbGry8AgM'
]

def download_audio(vid):
    out = f'/kaggle/working/audio/{vid}.wav'
    if os.path.exists(out): return out
    cmd = ['yt-dlp', '-f', 'bestaudio[ext=m4a]',
            '--extract-audio', '--audio-format', 'wav',
            '-o', f'/kaggle/working/audio/{vid}.%(ext)s',
            f'https://www.youtube.com/watch?v={vid}',
            '--no-playlist', '--quiet', '--socket-timeout', '60']
    try:
        subprocess.run(cmd, capture_output=True, timeout=120)
        # Rename if needed
        for ext in ['m4a', 'webm', 'mp4']:
            tmp = f'/kaggle/working/audio/{vid}.{ext}'
            if os.path.exists(tmp) and tmp != out:
                os.rename(tmp, out)
        return out if os.path.exists(out) else None
    except:
        return None

def extract_features(audio_path, vid):
    """Extract WavLM + prosody features"""
    try:
        # Load audio
        y, sr = librosa.load(audio_path, sr=16000, mono=True)
        
        # Split into 5-second chunks
        chunk_size = 16000 * 5
        n_chunks = len(y) // chunk_size
        
        if n_chunks == 0:
            print(f'  {vid}: Audio too short')
            return None
        
        features = []
        prosody_features = []
        
        for i in range(n_chunks):
            chunk = y[i*chunk_size:(i+1)*chunk_size]
            
            # WavLM features
            chunk_t = torch.tensor(chunk).unsqueeze(0).cuda()
            with torch.no_grad():
                rep = wavlm(chunk_t).last_hidden_state
            wavlm_feat = rep.mean(dim=2).squeeze().cpu().numpy()
            
            # Prosody features
            prosody = extract_prosody(chunk, 16000)
            
            combined = np.concatenate([wavlm_feat, prosody])
            features.append(combined)
        
        return np.array(features)
        
    except Exception as e:
        print(f'  {vid}: Error - {e}')
        return None

def extract_prosody(y, sr):
    """Extract 23-dim prosody features"""
    feats = []
    
    # F0 (5 dims)
    try:
        f0, voiced, _ = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
        f0_clean = f0[~np.isnan(f0)]
        voiced = voiced[~np.isnan(f0)]
        feats.extend([
            np.mean(f0_clean) if len(f0_clean) > 0 else 0,
            np.std(f0_clean) if len(f0_clean) > 0 else 0,
            np.max(f0_clean) if len(f0_clean) > 0 else 0,
            np.min(f0_clean) if len(f0_clean) > 0 else 0,
            np.mean(voiced) if len(voiced) > 0 else 0
        ])
    except:
        feats.extend([0] * 5)
    
    # Energy (5 dims)
    hop = 512
    rms = librosa.feature.rms(y=y, hop_length=hop)[0]
    feats.extend([np.mean(rms), np.std(rms), np.max(rms), np.min(rms), np.max(rms)-np.min(rms)])
    
    # Duration (2 dims)
    dur = len(y) / sr
    speech_rate = dur / (np.sum(rms > np.mean(rms)) + 1)
    feats.extend([dur, speech_rate])
    
    # Spectral (5 dims)
    try:
        sc = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0]
        sb = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=hop)[0]
        sf = librosa.feature.spectral_flatness(y=y, hop_length=hop)[0]
        zcr = librosa.feature.zero_crossing_rate(y, hop_length=hop)[0]
        feats.extend([np.mean(sc), np.mean(sb), np.mean(sf), np.mean(zcr), np.std(zcr)])
    except:
        feats.extend([0] * 5)
    
    # Voice quality (6 dims)
    try:
        y_harm, _ = librosa.effects.hpss(y)
        hnr = np.mean(np.abs(y_harm)) / (np.mean(np.abs(y)) + 1e-8)
        feats.extend([hnr, np.mean(np.abs(y)), np.std(y), np.max(np.abs(y)), 0, 0])
    except:
        feats.extend([0] * 6)
    
    return np.array(feats[:23], dtype=np.float32)

# Process first video
vid = VIDEO_IDS[0]
print(f'Processing: {vid}')
audio_path = download_audio(vid)
if audio_path:
    print(f'  Downloaded: {audio_path}')
    features = extract_features(audio_path, vid)
    if features is not None:
        np.save(f'/kaggle/working/features/{vid}_features.npy', features)
        print(f'  Saved: {features.shape}')
else:
    print(f'  Download failed')

In [ ]:
# Cell 5: Summary
import os
feat_files = [f for f in os.listdir('/kaggle/working/features') if f.endswith('.npy')]
print(f'Features extracted: {len(feat_files)}')
print('Done!')